
# Task 4 — Sentiment Analysis (CODTECH Internship)

**Deliverable:** Notebook demonstrating data preprocessing, model implementation, evaluation, and insights.

This notebook was generated automatically and uses a synthetic dataset saved as `sentiment_reviews_task4.csv`.


In [ ]:

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import nltk
import re
import joblib

# download NLTK resources (if not present)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [ ]:
# Load data
df = pd.read_csv(r"/mnt/data/sentiment_reviews_task4.csv")
print('Dataset shape:', df.shape)
df.head(5)

In [ ]:

# Basic EDA
print('Sentiment distribution:\n', df['sentiment'].value_counts())
plt.figure(figsize=(6,4))
sns.countplot(x='sentiment', data=df)
plt.title('Sentiment Distribution (0 = negative, 1 = positive)')
plt.show()


In [ ]:

# Text preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # lower
    text = str(text).lower()
    # remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # tokenize and remove stopwords, lemmatize
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens if tok not in stop_words and len(tok) > 1]
    return ' '.join(tokens)

# Apply (show sample)
df['clean_text'] = df['review_text'].apply(preprocess_text)
df[['review_text','clean_text']].head(6)


In [ ]:

# Vectorization (TF-IDF)
X = df['clean_text']
y = df['sentiment']

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_vec = tfidf.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


In [ ]:

# Train Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('\nClassification Report for Logistic Regression:\n', classification_report(y_test, y_pred_lr))
cm_lr = confusion_matrix(y_test, y_pred_lr)

# Train Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print('Naive Bayes Accuracy:', accuracy_score(y_test, y_pred_nb))
print('\nClassification Report for Naive Bayes:\n', classification_report(y_test, y_pred_nb))
cm_nb = confusion_matrix(y_test, y_pred_nb)

# Plot confusion matrices
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('LR Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')

plt.subplot(1,2,2)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Greens')
plt.title('NB Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()


In [ ]:

# Top features for Logistic Regression
feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]
top_positive_indices = np.argsort(coefs)[-20:][::-1]
top_negative_indices = np.argsort(coefs)[:20]

print('Top positive indicators:')
for idx in top_positive_indices[:15]:
    print(feature_names[idx], round(coefs[idx], 4))

print('\nTop negative indicators:')
for idx in top_negative_indices[:15]:
    print(feature_names[idx], round(coefs[idx], 4))


In [ ]:

# Word clouds for positive and negative (optional - requires wordcloud package)
try:
    from wordcloud import WordCloud
    pos_text = ' '.join(df[df['sentiment']==1]['clean_text'].tolist())
    neg_text = ' '.join(df[df['sentiment']==0]['clean_text'].tolist())

    wc_pos = WordCloud(width=600, height=400).generate(pos_text)
    wc_neg = WordCloud(width=600, height=400).generate(neg_text)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.imshow(wc_pos, interpolation='bilinear'); plt.axis('off'); plt.title('Positive Reviews WordCloud')
    plt.subplot(1,2,2)
    plt.imshow(wc_neg, interpolation='bilinear'); plt.axis('off'); plt.title('Negative Reviews WordCloud')
    plt.show()
except Exception as e:
    print('WordCloud not generated (wordcloud library may be missing). Error:', e)


In [ ]:

# Save trained model and TF-IDF vectorizer for later use
import joblib
joblib.dump(lr, '/mnt/data/sentiment_lr_model_task4.joblib')
joblib.dump(tfidf, '/mnt/data/sentiment_tfidf_task4.joblib')
print('Saved model and vectorizer to /mnt/data/')


In [ ]:

# Quick inference function
def predict_sentiment(text):
    cleaned = preprocess_text(text)
    vec = tfidf.transform([cleaned])
    pred = lr.predict(vec)[0]
    prob = lr.predict_proba(vec).max()
    return pred, prob

examples = [
    "I absolutely loved this — such a beautiful movie!",
    "That was a horrible film, I want my time back."
]
for ex in examples:
    print(ex, '->', predict_sentiment(ex))



## Conclusions & Insights

- The notebook demonstrates a full sentiment analysis pipeline: preprocessing, TF-IDF vectorization, model training and evaluation.
- On this synthetic dataset, Logistic Regression / Naive Bayes typically perform well.
- The notebook saves the trained model and vectorizer for reuse.
- To improve with real-world data: use a larger labeled dataset (e.g., IMDb, Twitter), consider fine-tuned transformer models (BERT), and add more advanced preprocessing (negation handling, emoji processing).

**Files created:** `/mnt/data/sentiment_reviews_task4.csv`, `/mnt/data/sentiment_analysis_task4.ipynb`, model files in `/mnt/data/`.
